# NVIDIA Synthetic Video Detector - Python Notebook

This notebook demonstrates how to use the NVIDIA Synthetic Video Detector service with Python. The service analyzes videos to detect whether they are synthetic/fake or real using a DINO-based deep learning model.

## Overview

The service processes video files and provides detection results with probability scores:

- **Service Integration**: Connect to NVIDIA Synthetic Video Detector services
- **Video Analysis**: Detect synthetic/fake content in videos
- **Per-frame Results**: Get detailed detection scores for each video frame
- **Visualization**: Plot probability scores over time with threshold lines
- **CSV Export**: Save detailed results for further analysis

## Requirements

- **Input Video**: Supports MP4 files with H.264 video codec (audio optional). Videos with Variable Frame Rate (VFR) are not supported.
- **Output**: Detection verdict (REAL vs SYNTHETIC/FAKE) with confidence scores
- **Service**: Access to a running NVIDIA Synthetic Video Detector service instance


## Installation

### Requirements:
- Python 3.10+ or later
- pip package manager
- gRPC and matplotlib dependencies from the requirements.txt file

Run the cell below to install all required packages:


In [ ]:
%pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your NVIDIA Synthetic Video Detector service. The service can be running on your machine or on a remote server accessible from your environment.


In [ ]:
import os
import sys
import pathlib

# Setup paths for NVIDIA Synthetic Video Detector modules
SCRIPT_PATH = str(pathlib.Path().resolve())
print(f"Notebook path: {SCRIPT_PATH}")

sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../"))

# Service connection configuration
SERVICE_HOST = "localhost"  # Update to your service host
SERVICE_PORT = 8001         # Update to your service port
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print(f"Python paths configured for NVIDIA Synthetic Video Detector modules")


## Import Libraries

Import the required libraries for gRPC communication, visualization, and the NVIDIA Synthetic Video Detector service modules. This cell also defines a sigmoid utility function used to convert model logits into probability scores.

In [ ]:
import grpc
import time
import math
from typing import Iterator, List, Tuple
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64
from io import BytesIO

# Import NVIDIA Synthetic Video Detector modules
from constants import DATA_CHUNK_SIZE
import syntheticvideodetector_pb2
import syntheticvideodetector_pb2_grpc


def sigmoid(x: float) -> float:
    """Convert logit to probability using sigmoid function.
    
    Args:
        x: Logit value
        
    Returns:
        Probability value between 0 and 1
    """
    try:
        return 1.0 / (1.0 + math.exp(-x))
    except OverflowError:
        return 0.0 if x < 0 else 1.0


print("All libraries imported successfully!")

## Helper Functions

Define functions for processing video data and communicating with the NVIDIA Synthetic Video Detector service. These functions handle the bi-directional gRPC streaming protocol used by the service.

### Function Overview

The NVIDIA Synthetic Video Detector service uses bi-directional gRPC streaming:

1. **Request Generator**: Streams video data in chunks to the service
2. **Response Processor**: Processes incoming detection results (per-frame results, final results, keepalive messages)
3. **Visualization**: Plots probability scores over time with threshold lines


In [ ]:
def fmt_elapsed(sec: float) -> str:
    """Format elapsed time in MM:SS format.
    
    Args:
        sec: Elapsed time in seconds
        
    Returns:
        Formatted string in MM:SS format
    """
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"


def generate_request_for_inference(
    video_filepath: str,
    chunk_size: int = DATA_CHUNK_SIZE
) -> Iterator[syntheticvideodetector_pb2.DetectSyntheticVideoRequest]:
    """Generate streaming requests for the NVIDIA Synthetic Video Detector service.
    
    This function creates a Python iterator that yields gRPC request messages
    containing video data chunks for streaming to the detection service.
    
    Args:
        video_filepath: Path to the input video file
        chunk_size: Size of data chunks to send (default: 1MB)
    
    Yields:
        DetectSyntheticVideoRequest messages containing video data chunks
        
    Raises:
        FileNotFoundError: If input file doesn't exist
        IOError: If input file cannot be read
    """
    
    # Validate input file exists
    if not os.path.exists(video_filepath):
        raise FileNotFoundError(f"Video file not found: {video_filepath}")
    
    file_size = os.path.getsize(video_filepath)
    print(f"Video file size: {file_size / (1024*1024):.2f} MB")
    
    # Send video data in chunks with progress tracking
    try:
        with open(video_filepath, "rb") as video_file:
            while True:
                chunk = video_file.read(chunk_size)
                if not chunk:
                    break
                yield syntheticvideodetector_pb2.DetectSyntheticVideoRequest(
                    video_file_data=chunk
                )
        print("Upload complete!")
    except IOError as e:
        raise IOError(f"Failed to read video file: {e}")


def process_detection_response(
    response_iter: Iterator[syntheticvideodetector_pb2.DetectSyntheticVideoResponse],
    show_progress: bool = True
) -> Tuple[List[dict], dict]:
    """Process detection responses from the service.
    
    Args:
        response_iter: Iterator of response messages from the service
        show_progress: Whether to show progress updates
    
    Returns:
        Tuple of (frame_results, final_result) where:
        - frame_results: List of dicts with 'index', 'logit', and 'probability' for each frame
        - final_result: Dict with final aggregated results
    """
    
    frame_results = []
    final_result = None
    start_time = time.time()
    
    if show_progress:
        print("Processing detection responses...")
    
    try:
        for response in response_iter:
            if response.HasField("clip_result"):
                # Store frame result with calculated probability
                frame = response.clip_result
                probability = sigmoid(frame.logit)  # Convert logit to probability
                frame_results.append({
                    'index': frame.index,
                    'logit': frame.logit,
                    'probability': probability
                })
                
            elif response.HasField("final_result"):
                final_result = {
                    'logit': response.final_result.logit,
                    'probability': response.final_result.probability,
                    'total_frames': response.final_result.total_clips,
                    'csv_data': response.final_result.csv_data
                }
                
            elif response.HasField("keepalive"):
                # Keepalive message - continue processing
                pass
        if show_progress and frame_results:
            elapsed = fmt_elapsed(time.time() - start_time)
            print(f"Processed {len(frame_results)} frames in {elapsed}")
    except Exception as e:
        print(f"\nError processing responses: {e}")
        raise
    
    return frame_results, final_result


print("Helper functions defined successfully!")

## Visualization Function

Create a function to plot probability scores over time with a threshold line to visualize the detection results.


In [ ]:
def plot_detection_results(
    clip_results: List[dict],
    final_result: dict,
    video_filepath: str = None,
    threshold: float = 0.3,
    figsize: tuple = (14, 6),
    video_name: str = "Video"
):
    """Plot per-frame probability scores over time and display the video being tested.
    
    Args:
        clip_results: List of dicts with 'index', 'logit', and 'probability' per frame
        final_result: Dict with final aggregated results
        video_filepath: Path to the video file to display
        threshold: Threshold value for synthetic/real classification (default: 0.3)
        figsize: Figure size as (width, height) tuple
        video_name: Name of the video for the plot title
    """
    
    if not clip_results:
        print("No frame results to plot")
        return
    
    # Extract data for plotting
    indices = [clip['index'] for clip in clip_results]
    probabilities = [clip['probability'] for clip in clip_results]
    
    # Use frame indices for x-axis
    x_data = indices
    x_label = 'Frame Index (Frame Number)'
    # Create figure with one subplot
    fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))
    
    # Plot: Probability over time
    ax1.plot(x_data, probabilities, 'b-', linewidth=2, label='Probability', alpha=0.7)
    ax1.axhline(y=threshold, color='r', linestyle='--', linewidth=2, 
                label=f'Threshold ({threshold})')
    
    # Add final probability as horizontal line
    if final_result:
        ax1.axhline(y=final_result['probability'], color='g', linestyle='-', 
                   linewidth=2, alpha=0.5, label=f"Final Prob ({final_result['probability']:.3f})")
    
    # Fill areas above and below threshold
    ax1.fill_between(x_data, probabilities, threshold, 
                     where=[p >= threshold for p in probabilities],
                     alpha=0.3, color='red', label='Synthetic region')
    ax1.fill_between(x_data, probabilities, threshold,
                     where=[p < threshold for p in probabilities],
                     alpha=0.3, color='green', label='Real region')
    
    ax1.set_xlabel(x_label, fontsize=12)
    ax1.set_ylabel('Probability (Synthetic)', fontsize=12)
    ax1.set_title(f'Detection Probability Over Time - {video_name}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best')
    ax1.set_ylim([0, 1])
    
    plt.tight_layout()
    
    # Save plot to BytesIO buffer and encode as base64
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plot_base64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close()
    
    # Display plot and video side by side
    if video_filepath and os.path.exists(video_filepath):
        print(f"\n{'='*60}")
        print(f"VIDEO BEING TESTED: {os.path.basename(video_filepath)}")
        print(f"{'='*60}\n")
        
        # Create HTML for side-by-side layout
        html_content = f"""
        <div style="display: flex; flex-wrap: wrap; gap: 20px; align-items: flex-start;">
            <div style="flex: 1; min-width: 500px;">
                <img src="data:image/png;base64,{plot_base64}" style="width: 100%; height: auto;">
            </div>
            <div style="flex: 1; min-width: 500px;">
                <video width="100%" height="auto" controls>
                    <source src="{video_filepath}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
            </div>
        </div>
        """
        display(HTML(html_content))
    else:
        # If no video, just show the plot
        plt.show()
    
    # Print statistics
    print("\n" + "="*60)
    print("DETECTION STATISTICS")
    print("="*60)
    print(f"Total frames analyzed: {len(clip_results)}")
    print(f"Average probability: {sum(probabilities) / len(probabilities):.4f}")
    
    # Count frames above and below threshold
    above_threshold = sum(1 for p in probabilities if p >= threshold)
    below_threshold = len(probabilities) - above_threshold
    
    print(f"\nFrames above threshold ({threshold}): {above_threshold} ({above_threshold/len(probabilities)*100:.1f}%)")
    print(f"Frames below threshold ({threshold}): {below_threshold} ({below_threshold/len(probabilities)*100:.1f}%)")
    
    if final_result:
        verdict = "SYNTHETIC/FAKE" if final_result['probability'] > threshold else "REAL"
        confidence = final_result['probability'] * 100 if final_result['probability'] > threshold else (1 - final_result['probability']) * 100
        
        print("\n" + "*"*60)
        print(f"FINAL VERDICT: {verdict}")
        print(f"CONFIDENCE: {confidence:.2f}%")
        print("*"*60)


print("Visualization function defined successfully!")

## Main Detection Function

Combine all helper functions into a single detection pipeline that analyzes a video and optionally plots the results.


In [ ]:
def detect_synthetic_video(
    video_filepath: str,
    server_target: str = SERVICE_TARGET,
    save_csv: bool = False,
    output_csv: str = None,
    plot_results: bool = True,
    threshold: float = 0.3
) -> Tuple[List[dict], dict]:
    """Complete pipeline to detect synthetic video content.
    
    Args:
        video_filepath: Path to video file to analyze
        server_target: gRPC server address (host:port)
        save_csv: Whether to save detailed CSV results
        output_csv: Path for output CSV file (default: auto-generated)
        plot_results: Whether to plot visualization
        threshold: Threshold for classification (default: 0.3)
    
    Returns:
        Tuple of (clip_results, final_result)
    """
    
    print("="*60)
    print("SYNTHETIC VIDEO DETECTOR")
    print("="*60)
    print(f"Video: {video_filepath}")
    print(f"Server: {server_target}")
    print("="*60)
    print()
    
    # Create gRPC channel
    with grpc.insecure_channel(server_target) as channel:
        # Create stub
        stub = syntheticvideodetector_pb2_grpc.SyntheticVideoDetectorServiceStub(channel)
        
        # Generate requests
        request_generator = generate_request_for_inference(video_filepath)
        
        # Call service and process responses
        response_iter = stub.DetectSyntheticVideo(request_generator)
        clip_results, final_result = process_detection_response(response_iter)
    
    print("\n" + "="*60)
    print("DETECTION COMPLETE")
    print("="*60)
    
    # Print results
    if final_result:
        print(f"\nTotal frames processed: {final_result['total_frames']}")
        print(f"Final logit: {final_result['logit']:.6f}")
        print(f"Final probability: {final_result['probability']:.6f}")
        
        verdict = "SYNTHETIC/FAKE" if final_result['probability'] > threshold else "REAL"
        confidence = final_result['probability'] * 100 if final_result['probability'] > threshold else (1 - final_result['probability']) * 100
        
        print(f"\nVERDICT: {verdict} (confidence: {confidence:.2f}%)")
        
        # Save CSV if requested (write index,probability)
        if save_csv and final_result.get('csv_data'):
            if output_csv is None:
                video_name = os.path.splitext(os.path.basename(video_filepath))[0]
                output_csv = f"detection_results_{video_name}.csv"

            # Transform server CSV (index,logit) into (index,probability)
            import math, csv
            rows = []
            for ln in final_result['csv_data'].strip().splitlines():
                parts = [p.strip() for p in ln.split(',')]
                if len(parts) < 2:
                    continue
                try:
                    idx = int(parts[0]); logit = float(parts[1])
                except ValueError:
                    continue
                try:
                    p = 1.0 / (1.0 + math.exp(-logit))
                except OverflowError:
                    p = 0.0 if logit < 0 else 1.0
                rows.append((idx, p))
            if rows:
                with open(output_csv, 'w', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(['index', 'probability'])
                    for idx, p in rows:
                        writer.writerow([idx, f'{p:.6f}'])
                print(f"\nCSV results saved to: {output_csv}")
    
    # Plot results if requested
    if plot_results and clip_results:
        video_name = os.path.basename(video_filepath)
        plot_detection_results(clip_results, final_result, video_filepath=video_filepath, threshold=threshold, video_name=video_name)
    
    return clip_results, final_result


print("Main detection function defined successfully!")

## Example Usage

Now you can use the detection function to analyze videos. Update the video path below to analyze your own videos.


In [ ]:
# Example 1: Analyze a video with visualization and default threshold (0.3)
VIDEO_PATH = "../assets/fake_sample_video.mp4"  # Update with your video path

frame_results, final_result = detect_synthetic_video(
    video_filepath=VIDEO_PATH,
    server_target=SERVICE_TARGET,
    plot_results=True,
    threshold=0.3
)

## Understanding the Visualization

### 1. Probability Over Time (Left Plot)
- **Blue line**: Per-frame probability score (higher = more likely synthetic)
- **Red dashed line**: Classification threshold (default: 0.3)
- **Green horizontal line**: Final aggregated probability
- **Red shaded area**: Regions where frames are classified as SYNTHETIC (above threshold)
- **Green shaded area**: Regions where frames are classified as REAL (below threshold)

### 2. Video Display (Right Side)
- **Video Player**: The analyzed video is displayed for visual review
- **Playback Controls**: Use the video player to review specific segments
- **Visual Inspection**: Compare the video content with the detection scores to understand which parts may be synthetic

### Key Insights:
- **Consistent high probabilities**: Likely a synthetic video throughout
- **Consistent low probabilities**: Likely a real video throughout
- **Mixed probabilities**: May indicate:
  - Partially synthetic video (some frames edited)
  - Scene changes affecting detection
  - Lower confidence regions requiring manual review

### Threshold Selection:
- **0.3 (default)**: The 0.3 threshold is chosen, being more aggressive in classifying synthetic videos correctly as synthetic.
